In [1]:
import pandas as pd
import numpy as np
import os
from Bio import Entrez,SeqIO
import warnings
warnings.filterwarnings("ignore")
from Bio.Blast.Applications import NcbimakeblastdbCommandline, NcbiblastpCommandline, NcbiblastnCommandline,NcbiblastxCommandline

### Step 1. make database

In [2]:
#C:\Program Files\NCBI\blast-2.16.0+\bin
from Bio.Blast.Applications import NcbimakeblastdbCommandline
import subprocess
import os

# 定义文件路径
database_fasta_file = "D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\data\\oriT_all.fasta"

# BLAST+工具的绝对路径
makeblastdb_path = "C:\\Program Files\\NCBI\\blast-2.16.0+\\bin\\makeblastdb.exe"

# 创建BLAST数据库
makeblastdb_cmd = NcbimakeblastdbCommandline(
    cmd=makeblastdb_path,
    dbtype="nucl",
    input_file=database_fasta_file
)

try:
    # 运行命令并捕获输出
    process = subprocess.Popen(
        str(makeblastdb_cmd),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        shell=True,
        text=True,  # 设置为文本模式
        encoding='utf-8'  # 指定编码为utf-8
    )
    stdout, stderr = process.communicate()

    # 打印输出
    print("标准输出:\n", stdout)
    print("标准错误输出:\n", stderr)

    # 检查返回码
    if process.returncode != 0:
        print(f"makeblastdb 命令返回非零退出代码: {process.returncode}")
        print("标准错误输出:\n", stderr)

except subprocess.CalledProcessError as e:
    print("子进程调用错误:", e)
    print("返回代码:", e.returncode)
    print("标准输出:\n", e.stdout)
    print("标准错误输出:\n", e.stderr)
except Exception as e:
    print("其他错误:", e)

print("BLAST数据库创建完成")

标准输出:
 

Building a new DB, current time: 11/25/2024 17:16:07
New DB name:   D:\19.TE_HT\06.HT_Family\07.OriT_DB\data\oriT_all.fasta
New DB title:  D:\19.TE_HT\06.HT_Family\07.OriT_DB\data\oriT_all.fasta
Sequence type: Nucleotide
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 23527 sequences in 0.981455 seconds.



标准错误输出:
 
BLAST数据库创建完成


### Step 2. Blastn

In [ ]:
# 定义文件路径
database_fasta_file ='D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\data\\oriT_all.fasta'
seq_path='D:\\19.TE_HT\\06.HT_Family\\02.Family_Fasta\\'
blastn_path="C:\\Program Files\\NCBI\\blast-2.16.0+\\bin\\blastn.exe"
files=os.listdir(seq_path)
for file in files:
    query_fasta_file =seq_path+file
    blast_output_file = 'D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\blastn\\'+file.replace(".fa",'')+"_blastn_results.csv"
    # 自定义输出格式字符串
    custom_outfmt = '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore'
    # 运行BLAST比对
    blastn_cmd = NcbiblastnCommandline(
            cmd=blastn_path,
            query=query_fasta_file,
            db=database_fasta_file,
            outfmt=custom_outfmt,
            evalue=1e-5,
            max_target_seqs=5,
            out=blast_output_file,
            num_threads=2 # 使用4个线程
        )
    blastn_cmd()
    #print("BLAST比对完成，结果已保存到:\t"+file.replace(".fa",'')+"_blastn_results.csv")
    r=open(blast_output_file,'r').readlines()
    print(file,"Blast Result Count:\t",len(r))

In [18]:
root_seq_path='D:\\19.TE_HT\\00.Sequence\\TEpep\\'
families=os.listdir(root_seq_path)
families

['CACTA',
 'Copia',
 'Gypsy',
 'hAT',
 'Helitron',
 'LINE',
 'LTR_Roo',
 'Mariner',
 'Mutator',
 'new_other',
 'Other']

In [ ]:
# 定义文件路径
database_fasta_file ='D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\data\\oriT_all.fasta'
root_seq_path='D:\\19.TE_HT\\00.Sequence\\TEpep\\'#'D:\\19.TE_HT\\00.Sequence\\ncRNA\\'#'D:\\19.TE_HT\\06.HT_Family\\02.Family_Fasta\\'
blastn_path="C:\\Program Files\\NCBI\\blast-2.16.0+\\bin\\blastn.exe"
families=os.listdir(root_seq_path)
for family in families[-1:]:
    print(family)
    blastn_out_path='D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\blastn\\'+"TEpep_"+family
    if "TEpep_"+family not in os.listdir('D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\blastn\\'):
        os.mkdir(blastn_out_path)
    seq_path=root_seq_path+family+"\\"
    files=os.listdir(seq_path)
    for file in files:
        out_file=file.replace(".fa",'')+"_blastn_results.csv"
        if out_file not in os.listdir(blastn_out_path):
            query_fasta_file =seq_path+file
            blast_output_file = blastn_out_path+"\\"+out_file
            # 自定义输出格式字符串
            custom_outfmt = '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore'
            # 运行BLAST比对
            blastn_cmd = NcbiblastnCommandline(
                    cmd=blastn_path,
                    query=query_fasta_file,
                    db=database_fasta_file,
                    outfmt=custom_outfmt,
                    evalue=1e-5,
                    max_target_seqs=5,
                    out=blast_output_file,
                    num_threads=2 # 使用4个线程
                )
            blastn_cmd()
            #print("BLAST比对完成，结果已保存到:\t"+file.replace(".fa",'')+"_blastn_results.csv")
            r=open(blast_output_file,'r').readlines()
            if len(r)>0:
                print(files.index(file),file,"Blast Result Count:\t",len(r))
            else:
                print(files.index(file))

### Step 3.Blastx

In [1]:
import pandas as pd
import numpy as np
import os
from Bio import Entrez,SeqIO
Entrez.email = "huangyan8@genomics.cn"  # 设置你的邮箱地址

from collections import Counter
import warnings
warnings.filterwarnings("ignore")
from Bio.Blast.Applications import NcbimakeblastdbCommandline, NcbiblastpCommandline, NcbiblastnCommandline,NcbiblastxCommandline
# 定义文件路径

In [2]:
root_seq_path='D:\\19.TE_HT\\00.Sequence\\TE\\'#'D:\\19.TE_HT\\00.Sequence\\ncRNA\\'#'D:\\19.TE_HT\\06.HT_Family\\02.Family_Fasta\\'
blastx_path="C:\\Program Files\\NCBI\\blast-2.16.0+\\bin\\blastx.exe"
families=os.listdir(root_seq_path)
families

['CACTA',
 'Copia',
 'Gypsy',
 'hAT',
 'Helitron',
 'LINE',
 'Mariner',
 'Mutator',
 'Other',
 'PIF-Harbinger']

In [ ]:
for family in ['Gypsy']:
    print(family)
    blastx_out_path='D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\blastx\\'+"TE_"+family
    if "TE_"+family not in os.listdir('D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\blastx\\'):
        os.mkdir(blastx_out_path)
    seq_path=root_seq_path+family+"\\"
    files=os.listdir(seq_path)
    for file in files:
        print(file)
        for db in ['auxiliary','relaxase','t4cp']:
            database_fasta_file ='D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\data\\'+db+'_all.fasta'
            out_file=file.replace(".fa",'')+"_"+db+"_blastx_results.csv"
            if out_file not in os.listdir(blastx_out_path):
                query_fasta_file =seq_path+file
                blast_output_file = blastx_out_path+"\\"+out_file
                # 自定义输出格式字符串
                custom_outfmt = '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore'
                # 运行BLAST比对
                blastn_cmd = NcbiblastxCommandline(
                        cmd=blastx_path,
                        query=query_fasta_file,
                        db=database_fasta_file,
                        outfmt=custom_outfmt,
                        evalue=1e-5,
                        max_target_seqs=5,
                        out=blast_output_file,
                        num_threads=12 # 使用4个线程
                    )
                blastn_cmd()
                #print("BLAST比对完成，结果已保存到:\t"+file.replace(".fa",'')+"_blastn_results.csv")
                r=open(blast_output_file,'r').readlines()
                if len(r)>0:
                    print(files.index(file),file,"Blast Result Count:\t",len(r))
                else:
                    print(files.index(file))

### Step 4.Blastp
- Use oriTDB Protein Seqs to make databases
- Use DNA seqs Blastx 
- Use Protein Seqs Blastp

In [ ]:
#C:\Program Files\NCBI\blast-2.16.0+\bin
from Bio.Blast.Applications import NcbimakeblastdbCommandline
import subprocess
import os

# 定义文件路径
database_fasta_file = "D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\data\\auxiliary_all.fasta" #Protein Seq

# BLAST+工具的绝对路径
makeblastdb_path = "C:\\Program Files\\NCBI\\blast-2.16.0+\\bin\\makeblastdb.exe"

# 创建BLAST数据库
makeblastdb_cmd = NcbimakeblastdbCommandline(
    cmd=makeblastdb_path,
    dbtype="prot",
    input_file=database_fasta_file
)

try:
    # 运行命令并捕获输出
    process = subprocess.Popen(
        str(makeblastdb_cmd),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        shell=True,
        text=True,  # 设置为文本模式
        encoding='utf-8'  # 指定编码为utf-8
    )
    stdout, stderr = process.communicate()

    # 打印输出
    print("标准输出:\n", stdout)
    print("标准错误输出:\n", stderr)

    # 检查返回码
    if process.returncode != 0:
        print(f"makeblastdb 命令返回非零退出代码: {process.returncode}")
        print("标准错误输出:\n", stderr)

except subprocess.CalledProcessError as e:
    print("子进程调用错误:", e)
    print("返回代码:", e.returncode)
    print("标准输出:\n", e.stdout)
    print("标准错误输出:\n", e.stderr)
except Exception as e:
    print("其他错误:", e)

print("BLAST数据库创建完成")

In [ ]:
# BLAST+工具的绝对路径
makeblastdb_path = "C:\\Program Files\\NCBI\\blast-2.16.0+\\bin\\makeblastdb.exe"
blastp_path = "C:\\Program Files\\NCBI\\blast-2.16.0+\\bin\\blastp.exe"
protein_path='D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\data\\'

In [ ]:
# 定义文件路径
db_path='D:\\19.TE_HT\\06.HT_Family\\07.OriT_DB\\data\\'
protein_path='D:\\18.TE_Evolution\\02.Proteins_and_ncRNA\\Protein\\'
for sample in new:
    for db in ['auxiliary','relaxase','t4cp']:
        print(sample,db)
        database_fasta_file = db_path+db+"_all.fasta"
        query_fasta_file =protein_path+sample+ "_protein.fa"
        blast_output_file = './Result/'+sample+"_"+db+"_blastp_results.csv"
        # 自定义输出格式字符串
        custom_outfmt = '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore'
        # 运行BLAST比对
        blastp_cmd = NcbiblastpCommandline(
            cmd=blastp_path,
            query=query_fasta_file,
            db=database_fasta_file,
            outfmt=custom_outfmt,
            evalue=1e-5,
            max_target_seqs=5,
            out=blast_output_file,
            num_threads=12 # 使用4个线程
        )
        blastp_cmd()
        #print("BLAST比对完成，结果已保存到:\t"+sample+"_"+db+"_blastp_results.csv")
        r=open(blast_output_file,'r').readlines()
        print("Blast Result Count:\t",len(r))